# Sistema CRUD de Obras Publicas (RF01)

Interface em Python para listar, inserir, atualizar e excluir obras no PostgreSQL.
Utiliza `psycopg2` para escrita, `SQLAlchemy`/`Pandas` para consulta e `Panel` para a interface.

**Antes de executar:** configure o arquivo `python/.env` e execute o banco com Docker Compose.

In [ ]:
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import pandas as pd
import panel as pn
import psycopg2 as pg
from sqlalchemy import create_engine, text

# 1. Carrega variáveis do arquivo python/.env
ENV_PATH = os.path.join(os.getcwd(), ".env")
if not os.path.exists(ENV_PATH):
    ENV_PATH = os.path.join(os.getcwd(), "python", ".env")
load_dotenv(dotenv_path=ENV_PATH)

DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")

if not all((DB_HOST, DB_NAME, DB_USER, DB_PASS)):
    raise RuntimeError("Configure DB_HOST, DB_NAME, DB_USER e DB_PASS no arquivo python/.env")

# Conexão psycopg2 (escrita: INSERT, UPDATE, DELETE) e SQLAlchemy (leitura: SELECT)
con = pg.connect(host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASS)
cnx = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}"
engine = create_engine(cnx)

pn.extension("tabulator", notifications=True)


# 2. Funções auxiliares para carregar opções válidas de FKs (Secretaria e Licitação)
def carregar_opcoes_secretaria():
    df = pd.read_sql_query(
        "SELECT id_secretaria, COALESCE(sigla || ' - ', '') || nome_oficial AS label FROM secretaria ORDER BY id_secretaria",
        engine,
    )
    return dict(zip(df["label"], df["id_secretaria"]))


def carregar_opcoes_licitacao():
    df = pd.read_sql_query(
        "SELECT id_licitacao, numero_licitacao || ' (' || modalidade || ')' AS label FROM licitacao ORDER BY id_licitacao",
        engine,
    )
    opcoes = {"Nenhuma (Sem licitação vinculada)": None}
    opcoes.update(dict(zip(df["label"], df["id_licitacao"])))
    return opcoes


# 3. Widgets do formulário da tabela OBRA (RF01)
def create_form_widgets():
    return {
        "id_obra": pn.widgets.IntInput(
            name="ID da Obra (para Consultar, Atualizar ou Excluir)",
            value=0,
            start=0,
            sizing_mode="stretch_width",
        ),
        "titulo": pn.widgets.TextInput(
            name="Título da Obra *",
            placeholder="Ex: Construção de Quadra Poliesportiva",
            sizing_mode="stretch_width",
        ),
        "local_execucao": pn.widgets.TextInput(
            name="Local de Execução *",
            placeholder="Ex: Bairro Centro, Quixadá - CE",
            sizing_mode="stretch_width",
        ),
        "data_inicio": pn.widgets.DatePicker(name="Data de Início *"),
        "data_prevista_conclusao": pn.widgets.DatePicker(
            name="Data Prevista de Conclusão *"
        ),
        "orcamento_previsto": pn.widgets.FloatInput(
            name="Orçamento Previsto (R$) *",
            value=100000.00,
            step=1000.0,
            sizing_mode="stretch_width",
        ),
        "situacao": pn.widgets.Select(
            name="Situação da Obra *",
            options=[
                "planejada",
                "em andamento",
                "suspensa",
                "concluída",
                "cancelada",
            ],
            value="planejada",
            sizing_mode="stretch_width",
        ),
        "id_secretaria": pn.widgets.Select(
            name="Secretaria Responsável *",
            options=carregar_opcoes_secretaria(),
            sizing_mode="stretch_width",
        ),
        "id_licitacao": pn.widgets.Select(
            name="Licitação Vinculada (Opcional)",
            options=carregar_opcoes_licitacao(),
            sizing_mode="stretch_width",
        ),
    }


buttonConsultar = pn.widgets.Button(name="🔍 Consultar", button_type="primary")
buttonInserir = pn.widgets.Button(name="➕ Inserir", button_type="success")
buttonAtualizar = pn.widgets.Button(name="✏️ Atualizar", button_type="warning")
buttonExcluir = pn.widgets.Button(name="🗑️ Excluir", button_type="danger")


# 4. Operações CRUD com tratamento de erros e restrições de integridade
def queryAll(mensagem_sucesso=None):
    sql = """
        SELECT o.id_obra, o.titulo, o.local_execucao, o.data_inicio,
               o.data_prevista_conclusao, o.orcamento_previsto, o.situacao,
               s.sigla AS secretaria, l.numero_licitacao AS licitacao
        FROM obra o
        JOIN secretaria s ON s.id_secretaria = o.id_secretaria
        LEFT JOIN licitacao l ON l.id_licitacao = o.id_licitacao
        ORDER BY o.id_obra;
    """
    df = pd.read_sql_query(sql, engine)
    tabela = pn.widgets.Tabulator(
        df, show_index=False, sizing_mode="stretch_width", height=420
    )
    if mensagem_sucesso:
        return pn.Column(
            pn.pane.Alert(mensagem_sucesso, alert_type="success"), tabela
        )
    return tabela


def on_consultar(w):
    try:
        id_obra = w["id_obra"].value
        if not id_obra or id_obra <= 0:
            return queryAll()
        sql = text("""
            SELECT o.id_obra, o.titulo, o.local_execucao, o.data_inicio,
                   o.data_prevista_conclusao, o.orcamento_previsto, o.situacao,
                   s.sigla AS secretaria, l.numero_licitacao AS licitacao
            FROM obra o
            JOIN secretaria s ON s.id_secretaria = o.id_secretaria
            LEFT JOIN licitacao l ON l.id_licitacao = o.id_licitacao
            WHERE o.id_obra = :id_obra
        """)
        df = pd.read_sql_query(sql, engine, params={"id_obra": id_obra})
        if df.empty:
            return pn.pane.Alert(
                f"Nenhuma obra encontrada com ID {id_obra}.",
                alert_type="warning",
            )
        return pn.widgets.Tabulator(
            df, show_index=False, sizing_mode="stretch_width", height=420
        )
    except Exception as e:
        return pn.pane.Alert(f"Erro na consulta: {str(e)}", alert_type="danger")


def on_inserir(w):
    # Validação de campos obrigatórios antes de enviar ao banco (RN08)
    if (
        not (w["titulo"].value or "").strip()
        or not (w["local_execucao"].value or "").strip()
        or not w["data_inicio"].value
        or not w["data_prevista_conclusao"].value
    ):
        return pn.pane.Alert(
            "Erro de validação (RN08): Preencha todos os campos obrigatórios (*).",
            alert_type="danger",
        )
    if w["data_prevista_conclusao"].value < w["data_inicio"].value:
        return pn.pane.Alert(
            "Erro de validação: a data prevista não pode ser anterior à data de início.",
            alert_type="danger",
        )
    if not (w["orcamento_previsto"].value or "") or w["orcamento_previsto"].value <= 0:
        return pn.pane.Alert(
            "Erro de validação: o orçamento previsto deve ser maior que zero.",
            alert_type="danger",
        )
    if not w["id_secretaria"].value:
        return pn.pane.Alert(
            "Erro de validação: selecione uma secretaria responsável.",
            alert_type="danger",
        )
    try:
        with con.cursor() as cursor:
            cursor.execute(
                """
                INSERT INTO obra (
                    titulo, local_execucao, data_inicio, data_prevista_conclusao,
                    orcamento_previsto, situacao, id_secretaria, id_licitacao
                )
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (
                    w["titulo"].value.strip(),
                    w["local_execucao"].value.strip(),
                    w["data_inicio"].value,
                    w["data_prevista_conclusao"].value,
                    w["orcamento_previsto"].value,
                    w["situacao"].value,
                    w["id_secretaria"].value,
                    w["id_licitacao"].value,
                ),
            )
        con.commit()
        return queryAll("Obra cadastrada com sucesso!")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(
            f"Violação de restrição no banco ao inserir: {str(e)}",
            alert_type="danger",
        )


def on_atualizar(w):
    id_obra = w["id_obra"].value
    if not id_obra or id_obra <= 0:
        return pn.pane.Alert(
            "Informe um ID da Obra válido (> 0) para atualizar.",
            alert_type="warning",
        )
    if (
        not w["titulo"].value
        or not w["titulo"].value.strip()
        or not w["local_execucao"].value
        or not w["local_execucao"].value.strip()
        or not w["data_inicio"].value
        or not w["data_prevista_conclusao"].value
    ):
        return pn.pane.Alert(
            "Erro de validação (RN08): Título, local e datas são obrigatórios.",
            alert_type="danger",
        )
    if w["data_prevista_conclusao"].value < w["data_inicio"].value:
        return pn.pane.Alert(
            "Erro de validação: a data prevista não pode ser anterior à data de início.",
            alert_type="danger",
        )
    if not w["orcamento_previsto"].value or w["orcamento_previsto"].value <= 0:
        return pn.pane.Alert(
            "Erro de validação: o orçamento previsto deve ser maior que zero.",
            alert_type="danger",
        )
    if not w["id_secretaria"].value:
        return pn.pane.Alert(
            "Erro de validação: selecione uma secretaria responsável.",
            alert_type="danger",
        )
    try:
        with con.cursor() as cursor:
            cursor.execute(
                """
                UPDATE obra
                SET titulo = %s,
                    local_execucao = %s,
                    data_inicio = %s,
                    data_prevista_conclusao = %s,
                    orcamento_previsto = %s,
                    situacao = %s,
                    id_secretaria = %s,
                    id_licitacao = %s
                WHERE id_obra = %s
                """,
                (
                    w["titulo"].value.strip(),
                    w["local_execucao"].value.strip(),
                    w["data_inicio"].value,
                    w["data_prevista_conclusao"].value,
                    w["orcamento_previsto"].value,
                    w["situacao"].value,
                    w["id_secretaria"].value,
                    w["id_licitacao"].value,
                    id_obra,
                ),
            )
            afetados = cursor.rowcount
        con.commit()
        if afetados == 0:
            return pn.pane.Alert(
                f"Nenhuma obra encontrada com ID {id_obra} para atualizar.",
                alert_type="warning",
            )
        return queryAll(f"Obra ID {id_obra} atualizada com sucesso!")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(
            f"Violação de restrição no banco ao atualizar: {str(e)}",
            alert_type="danger",
        )


def on_excluir(w):
    id_obra = w["id_obra"].value
    if not id_obra or id_obra <= 0:
        return pn.pane.Alert(
            "Informe um ID da Obra válido (> 0) para excluir.",
            alert_type="warning",
        )
    try:
        with con.cursor() as cursor:
            cursor.execute("DELETE FROM obra WHERE id_obra = %s", (id_obra,))
            afetados = cursor.rowcount
        con.commit()
        if afetados == 0:
            return pn.pane.Alert(
                f"Nenhuma obra encontrada com ID {id_obra} para excluir.",
                alert_type="warning",
            )
        return queryAll(f"Obra ID {id_obra} excluída com sucesso!")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(
            f"Não foi possível excluir: {str(e)}", alert_type="danger"
        )


output_area = pn.Column()


def executar_acao(funcao):
    output_area.clear()
    output_area.append(funcao(form_widgets))


buttonConsultar.on_click(lambda e: executar_acao(on_consultar))
buttonInserir.on_click(lambda e: executar_acao(on_inserir))
buttonAtualizar.on_click(lambda e: executar_acao(on_atualizar))
buttonExcluir.on_click(lambda e: executar_acao(on_excluir))


# 5. Navegação Home / CRUD / Gráficos (mantendo o padrão visual da professora)
main_area = pn.Column()
form_widgets = create_form_widgets()

btn_go_crud = pn.widgets.Button(
    name="📋 Gerenciar Obras (CRUD)", button_type="primary", width=260
)
btn_go_graficos = pn.widgets.Button(
    name="📊 Orçamento por Situação", button_type="success", width=260
)


def create_btn_voltar():
    btn = pn.widgets.Button(name="⬅ Voltar para Home", button_type="default")
    btn.on_click(lambda e: go_to_home())
    return btn


def go_to_home():
    main_area.clear()
    main_area.append(
        pn.Column(
            pn.pane.Markdown("## 🏠 Módulo de Obras Públicas (RF01)"),
            pn.pane.Markdown(
                "Autor: **Guilherme Freires de Sousa** | Tabela: `obra`"
            ),
            pn.Spacer(height=15),
            btn_go_crud,
            btn_go_graficos,
            align="center",
            margin=(30, 30, 30, 30),
        )
    )


def go_to_crud():
    main_area.clear()
    output_area.clear()
    output_area.append(queryAll())
    main_area.append(
        pn.Row(
            pn.Column(
                pn.pane.Markdown("### 🏗️ Cadastro e Manutenção de Obras"),
                form_widgets["id_obra"],
                form_widgets["titulo"],
                form_widgets["local_execucao"],
                form_widgets["data_inicio"],
                form_widgets["data_prevista_conclusao"],
                form_widgets["orcamento_previsto"],
                form_widgets["situacao"],
                form_widgets["id_secretaria"],
                form_widgets["id_licitacao"],
                pn.Row(buttonConsultar, buttonInserir),
                pn.Row(buttonAtualizar, buttonExcluir),
                pn.Spacer(height=10),
                create_btn_voltar(),
                width=380,
                margin=(10, 20, 10, 10),
            ),
            pn.Column(output_area, sizing_mode="stretch_both"),
        )
    )

def grafico_orcamento_por_situacao():
    df = pd.read_sql_query(
        "SELECT situacao, SUM(orcamento_previsto) AS total_orcamento FROM obra GROUP BY situacao",
        engine,
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(
        df["situacao"],
        df["total_orcamento"],
        color=["#1f77b4", "#2ca02c", "#ff7f0e"],
    )
    ax.set_title("Orçamento Previsto Total por Situação da Obra")
    ax.set_ylabel("Valor Total (R$)")
    return pn.pane.Matplotlib(fig, tight=True)


def go_to_graficos():
    main_area.clear()
    main_area.append(
        pn.Column(
            pn.pane.Markdown("### 📊 Resumo Financeiro das Obras"),
            grafico_orcamento_por_situacao(),
            create_btn_voltar(),
            margin=(20, 20, 20, 20),
        )
    )


btn_go_crud.on_click(lambda e: go_to_crud())
btn_go_graficos.on_click(lambda e: go_to_graficos())

go_to_home()

pn.Column(
    pn.pane.Markdown("# 🏛️ Sistema de Acompanhamento de Obras Públicas"),
    main_area,
).servable()